<div style="background: linear-gradient(135deg,#5C1A05 0%,#BF360C 100%);
            padding:40px 36px 32px 36px;
            border-radius:12px;
            color:white;
            font-family:Arial,sans-serif">

<div style="font-size:13px; letter-spacing:2px; opacity:.75; margin-bottom:8px">
UNIVERSIDAD EXTERNADO DE COLOMBIA · CIENCIA DE DATOS · 2026
</div>

<h1 style="margin:0 0 6px 0; font-size:28px; font-weight:700; line-height:1.2">
Feature Engineering (Gold) — SPARCS (v2, Target Encoding)
</h1>

<h2 style="margin:0 0 20px 0; font-size:17px; font-weight:400; opacity:.9">
Split 80/20 y codificación de variables de alta cardinalidad
</h2>

<hr style="border:none; border-top:1px solid rgba(255,255,255,.3); margin:0 0 18px 0"/>

<div style="display:flex; gap:40px; font-size:13px; opacity:.85">

<div>
<b>Autores:</b><br>
Daniel Vallejo Vecil<br>
Juliana Muñoz González
</div>

<div>
<b>Director:</b><br>
Julián F. Sánchez López
</div>

<div>
<b>Capa construida:</b><br>
3_gold
</div>

</div>

</div>

<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
1. Objetivo
</h2>

</div>

<div style="background:#FBE9E7;
border-left:6px solid #BF360C;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">

Esta versión reemplaza el agrupamiento en <code>"OTROS"</code> (que concentraba entre 28% y 48% de
los registros, descartando información diagnóstica relevante) por <b>Target Encoding</b>: cada
categoría de <code>CCSR Diagnosis</code>, <code>CCSR Procedure</code> y <code>APR DRG</code> se
reemplaza por el promedio de <code>Length of Stay</code> de esa categoría, sin perder ninguna de las
478/321/334 categorías originales.

<b>Punto crítico:</b> esas medias se calculan <b>únicamente con el conjunto de entrenamiento</b>. Por
eso el split 80/20 se mueve a este notebook (antes vivía en <code>04_Modeling</code>) — el orden
correcto es dividir primero, calcular las medias solo con el 80% de entrenamiento, y aplicarlas a
ambos conjuntos. Si se calcularan con el dataset completo, el modelo vería indirectamente información
del conjunto de prueba a través de la codificación (fuga de información).

<b>Suavizado (shrinkage):</b> categorías con muy pocos casos (ej. un procedimiento extremadamente raro
con 1-2 casos en todo el entrenamiento) producen promedios poco confiables si se usan tal cual. Se
aplica suavizado tipo Bayesiano: el promedio de cada categoría se mezcla con el promedio global,
ponderado por cuántos casos la respaldan — las categorías con muchos casos casi no cambian, las raras
se acercan a la media global en vez de quedarse con un valor extremo no representativo.

<code>04_Modeling.ipynb</code> ya no necesita hacer el split — carga directamente
<code>X_train</code>, <code>X_test</code>, <code>y_train</code>, <code>y_test</code>.

</div>

<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
2. Configuración y carga
</h2>

</div>

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data"
SILVER_PATH = DATA_PATH / "2_silver"
GOLD_PATH = DATA_PATH / "3_gold"
GOLD_PATH.mkdir(parents=True, exist_ok=True)

TABLAS_PATH = PROJECT_ROOT / "tablas"
TABLAS_PATH.mkdir(parents=True, exist_ok=True)
NOTEBOOK_NUM = "03"


def guardar_tabla(tabla, nombre):
    if isinstance(tabla, pd.Series):
        tabla = tabla.to_frame()
    archivo = TABLAS_PATH / f"{NOTEBOOK_NUM}_{nombre}.csv"
    tabla.to_csv(archivo, encoding="utf-8-sig")
    print(f"Tabla guardada: {archivo.resolve()}")


df = pd.read_parquet(SILVER_PATH / "sparcs_silver.parquet")

print(f"Filas cargadas (Silver):    {df.shape[0]:,}")
print(f"Columnas cargadas (Silver): {df.shape[1]}")


Filas cargadas (Silver):    2,090,919
Columnas cargadas (Silver): 25


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
3. Eliminar el rastro residual de nulos (0.03%)
</h2>

</div>

In [2]:
n_antes = df.shape[0]
df = df.dropna(subset=["APR Risk of Mortality", "APR Severity of Illness Description"])
n_despues = df.shape[0]

print(f"Filas eliminadas: {n_antes - n_despues}")
print(f"Filas restantes: {n_despues:,}")


Filas eliminadas: 580
Filas restantes: 2,090,339


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
4. Combinar pares Código : Descripción
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Se combinan por legibilidad (útil más adelante para interpretar SHAP y las tablas de medias por
categoría), independientemente de si la variable luego se codifica con Target Encoding o One-Hot.


</div>

In [3]:
PARES_CODIGO_DESCRIPCION = [
    ("CCSR Diagnosis Code", "CCSR Diagnosis Description", "CCSR Diagnosis"),
    ("CCSR Procedure Code", "CCSR Procedure Description", "CCSR Procedure"),
    ("APR DRG Code", "APR DRG Description", "APR DRG"),
    ("APR MDC Code", "APR MDC Description", "APR MDC"),
    ("APR Severity of Illness Code", "APR Severity of Illness Description", "APR Severity of Illness"),
]

for col_codigo, col_desc, nombre_nuevo in PARES_CODIGO_DESCRIPCION:
    df[nombre_nuevo] = df[col_codigo].astype(str) + ": " + df[col_desc].astype(str)
    df = df.drop(columns=[col_codigo, col_desc])
    print(f"{nombre_nuevo}: {df[nombre_nuevo].nunique()} categorías combinadas")

print()
print(f"Columnas restantes: {df.shape[1]}")


CCSR Diagnosis: 471 categorías combinadas
CCSR Procedure: 320 categorías combinadas
APR DRG: 326 categorías combinadas
APR MDC: 24 categorías combinadas
APR Severity of Illness: 4 categorías combinadas

Columnas restantes: 20


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
5. Split 80/20 (ANTES de codificar)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


El split se hace ahora, con el dataframe todavía en su forma "cruda" (categorías como texto), porque
el Target Encoding de la siguiente sección necesita ver <code>Length of Stay</code> junto con las
categorías, pero <b>solo del lado de entrenamiento</b>.


</div>

In [4]:
TARGET = "Length of Stay"

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42)

print(f"Entrenamiento: {train_df.shape[0]:,} filas ({train_df.shape[0]/len(df)*100:.1f}%)")
print(f"Prueba:        {test_df.shape[0]:,} filas ({test_df.shape[0]/len(df)*100:.1f}%)")


Entrenamiento: 1,672,271 filas (80.0%)
Prueba:        418,068 filas (20.0%)


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
6. Target Encoding (solo variables de alta cardinalidad)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Para cada variable, se calcula la media de <code>Length of Stay</code> por categoría <b>usando
únicamente <code>train_df</code></b>. Esa tabla de medias se aplica tanto a <code>train_df</code> como
a <code>test_df</code>. Las categorías que aparecen en <code>test_df</code> pero nunca se vieron en
entrenamiento reciben la <b>media global de entrenamiento</b> (no se puede calcular una media
específica para algo que el modelo nunca vio).


</div>

In [5]:
VARIABLES_TARGET_ENCODING = ["CCSR Diagnosis", "CCSR Procedure", "APR DRG"]
K_SUAVIZADO = 20  # a mayor K, más se "jalan" hacia la media global las categorías con pocos casos

media_global_train = train_df[TARGET].mean()
tablas_encoding = {}

for var in VARIABLES_TARGET_ENCODING:
    if var not in train_df.columns:
        continue

    stats = train_df.groupby(var)[TARGET].agg(["mean", "count"])
    # Suavizado tipo Bayesiano/James-Stein: categorías con pocos casos se acercan a la media global
    stats["mean_suavizada"] = (
        (stats["count"] * stats["mean"] + K_SUAVIZADO * media_global_train)
        / (stats["count"] + K_SUAVIZADO)
    )
    tablas_encoding[var] = stats

    medias_dict = stats["mean_suavizada"]
    col_encoded = f"{var}_TE"
    train_df[col_encoded] = train_df[var].map(medias_dict)
    test_df[col_encoded] = test_df[var].map(medias_dict).fillna(media_global_train)

    n_no_vistas = test_df[var].isin(medias_dict.index).eq(False).sum()
    n_pocos_casos = (stats["count"] < K_SUAVIZADO).sum()
    print(f"{var}: {stats.shape[0]} categorías codificadas | "
          f"{n_no_vistas} registros de prueba con categoría no vista | "
          f"{n_pocos_casos} categorías con menos de {K_SUAVIZADO} casos (suavizadas hacia la media global)")

    train_df = train_df.drop(columns=[var])
    test_df = test_df.drop(columns=[var])

    guardar_tabla(stats.sort_values("mean_suavizada", ascending=False), f"TargetEncoding_{var.replace(' ', '')}")


CCSR Diagnosis: 470 categorías codificadas | 1 registros de prueba con categoría no vista | 53 categorías con menos de 20 casos (suavizadas hacia la media global)
Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_TargetEncoding_CCSRDiagnosis.csv
CCSR Procedure: 320 categorías codificadas | 0 registros de prueba con categoría no vista | 10 categorías con menos de 20 casos (suavizadas hacia la media global)
Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_TargetEncoding_CCSRProcedure.csv
APR DRG: 326 categorías codificadas | 0 registros de prueba con categoría no vista | 0 categorías con menos de 20 casos (suavizadas hacia la media global)
Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_TargetEncoding_APRDRG.csv


<div style="background:#FBE9E7;
border-left:6px solid #BF360C;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">

<h3 style="margin:0 0 10px 0;">6.1 Inspección de resultados: categorías con mayor y menor estancia promedio</h3>

Se muestran las 10 categorías con mayor y menor <code>Length of Stay</code> promedio (calculado solo
con entrenamiento) de cada variable — sirve como control de calidad (¿tiene sentido clínico?) y como
tabla directamente citable en la tesis.

</div>

In [6]:
for var, stats in tablas_encoding.items():
    print(f"===== {var}: TOP 10 mayor estancia promedio (suavizada) =====")
    display(stats.sort_values("mean_suavizada", ascending=False).head(10)
            [["count", "mean", "mean_suavizada"]])
    print()
    print(f"===== {var}: TOP 10 menor estancia promedio (suavizada) =====")
    display(stats.sort_values("mean_suavizada", ascending=True).head(10)
            [["count", "mean", "mean_suavizada"]])
    print()


===== CCSR Diagnosis: TOP 10 mayor estancia promedio (suavizada) =====


,count,mean,mean_suavizada
CCSR Diagnosis,,,
PNL002: Short gestation; low birth weight; and fetal growth retardation,242,26.330579,24.759113
PNL005: Respiratory distress syndrome,243,25.827160,24.299953
"INJ046: Spinal cord injury (SCI), subsequent encounter",139,26.719424,24.081054
NEO060: Leukemia - acute myeloid leukemia (AML),1244,22.450965,22.186620
NEO059: Leukemia - acute lymphoblastic leukemia (ALL),463,21.477322,20.825854
INF001: Tuberculosis,266,20.281955,19.265341
CIR022: Sequela of hemorrhagic cerebrovascular disease,938,18.295309,18.033286
PNL006: Respiratory perinatal condition,357,17.579832,16.951956
MBD001: Schizophrenia spectrum and other psychotic disorders,29447,16.263015,16.255876



===== CCSR Diagnosis: TOP 10 menor estancia promedio (suavizada) =====


,count,mean,mean_suavizada
CCSR Diagnosis,,,
END009: Obesity,16390,1.560220,1.565319
PNL007: Hemolytic jaundice and perinatal jaundice,1836,1.604575,1.649185
GEN020: Prolapse of female genital organs,707,1.581330,1.695856
PRG026: OB-related trauma to perineum and vulva,9335,1.960150,1.968240
RSP004: Acute and chronic tonsillitis,858,1.925408,2.012400
NEO029: Breast cancer - ductal carcinoma in situ (DCIS),494,1.949393,2.097057
GEN021: Menstrual disorders,398,1.974874,2.155233
GEN022: Benign ovarian cyst,677,2.078287,2.183483
PRG010: Hemorrhage after first trimester,333,1.993994,2.206480



===== CCSR Procedure: TOP 10 mayor estancia promedio (suavizada) =====


...

,count,mean,mean_suavizada
CCSR Procedure,,,
FRS009: CERVICAL CERCLAGE,340,1.505882,1.741354
CAR009: LEFT ATRIAL APPENDAGE PROCEDURES,1837,1.714752,1.758152
GIS010: GASTRECTOMY,12796,1.984761,1.990628
GNR001: COSMETIC PROCEDURES,219,1.698630,2.037186
MST008: ARTHROPLASTY OF OTHER JOINT (EXCLUDING KNEE AND HIP),3154,2.059290,2.082510
SKB003: MASTECTOMY AND LUMPECTOMY,1842,2.109121,2.148167
MRS003: PROSTATECTOMY,2626,2.150419,2.177584
ENT007: TONSILECTOMY AND ADENOIDECTOMY,576,2.140625,2.261556
PGN002: SPONTANEOUS VAGINAL DELIVERY,90809,2.287141,2.287902



===== APR DRG: TOP 10 mayor estancia promedio (suavizada) =====


,count,mean,mean_suavizada
APR DRG,,,
588: NEONATE BIRTH WEIGHT < 1500 GRAMS WITH MAJOR PROCEDURE,116,90.241379,77.815349
591: NEONATE BIRTH WEIGHT 500-749 GRAMS WITHOUT MAJOR PROCEDURE,217,81.230415,74.860285
593: NEONATE BIRTH WEIGHT 750-999 GRAMS WITHOUT MAJOR PROCEDURE,364,73.104396,69.596061
602: NEONATE BIRTH WEIGHT 1000-1249 GRAMS WITH RESPIRATORY DISTRESS SYNDROME OR OTHER MAJOR RESPIRATORY CON,423,58.345154,55.970401
4: TRACHEOSTOMY WITH MV >96 HOURS WITH EXTENSIVE PROCEDURE,2217,52.288227,51.872100
609: NEONATE BIRTH WEIGHT 1500-2499 GRAMS WITH MAJOR PROCEDURE,114,59.184211,51.208116
5: TRACHEOSTOMY WITH MV >96 HOURS WITHOUT EXTENSIVE PROCEDURE,3125,45.992000,45.736053
2: HEART AND/OR LUNG TRANSPLANT,334,46.386228,44.090078
607: NEONATE BIRTH WEIGHT 1250-1499 GRAMS WITH RESPIRATORY DISTRESS SYNDROME OR OTHER MAJOR RESPIRATORY CON,519,44.940270,43.485877



===== APR DRG: TOP 10 menor estancia promedio (suavizada) =====


,count,mean,mean_suavizada
APR DRG,,,
"581: NEONATE, TRANSFERRED < 5 DAYS OLD, BORN HERE",1177,1.274427,1.349112
403: PROCEDURES FOR OBESITY,16089,1.469389,1.474697
234: APPENDECTOMY WITHOUT COMPLEX PRINCIPAL DIAGNOSIS,4887,1.495191,1.512510
480: MAJOR MALE PELVIC PROCEDURES,1579,1.818873,1.867972
322: SHOULDER AND ELBOW JOINT REPLACEMENT,2710,1.933210,1.961131
"640: NEONATE BIRTH WEIGHT > 2499 GRAMS, NORMAL NEWBORN OR NEONATE WITH OTHER PROBLEM",134421,2.054619,2.055168
362: MASTECTOMY PROCEDURES,2315,2.044492,2.076183
203: CHEST PAIN,5489,2.095281,2.108529
97: TONSIL AND ADENOID PROCEDURES,587,2.013629,2.136553


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
7. One-Hot Encoding (variables de baja cardinalidad restantes)
</h2>

</div>

<div style="background:#F5F9FF;
border-left:6px solid #1565C0;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


Para evitar columnas dummy inconsistentes entre entrenamiento y prueba (que una categoría exista en
uno y no en el otro), se codifican juntas y luego se separan de nuevo — esto no genera fuga de
información porque el One-Hot solo usa la estructura de categorías, no la variable objetivo.


</div>

In [7]:
categoricas_restantes = train_df.drop(columns=[TARGET]).select_dtypes(include=["object"]).columns.tolist()
print(f"Variables categóricas a codificar con One-Hot: {len(categoricas_restantes)}")
print(categoricas_restantes)

train_df["_split_"] = "train"
test_df["_split_"] = "test"
combinado = pd.concat([train_df, test_df], axis=0)

combinado = pd.get_dummies(combinado, columns=categoricas_restantes, drop_first=True, dtype=np.int8)

train_df = combinado[combinado["_split_"] == "train"].drop(columns=["_split_"])
test_df = combinado[combinado["_split_"] == "test"].drop(columns=["_split_"])

print()
print(f"Columnas finales tras One-Hot: {train_df.shape[1]}")


C:\Users\danie\AppData\Local\Temp\ipykernel_4556\310716509.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas_restantes = train_df.drop(columns=[TARGET]).select_dtypes(include=["object"]).columns.tolist()


Variables categóricas a codificar con One-Hot: 16
['Hospital Service Area', 'Hospital County', 'Age Group', 'Zip Code - 3 digits', 'Gender', 'Race', 'Ethnicity', 'Type of Admission', 'APR Risk of Mortality', 'APR Medical Surgical Description', 'Payment Typology 1', 'Payment Typology 2', 'Payment Typology 3', 'Emergency Department Indicator', 'APR MDC', 'APR Severity of Illness']

Columnas finales tras One-Hot: 190


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
9. Separar X / y
</h2>

</div>

In [8]:
y_train = train_df[TARGET].copy()
X_train = train_df.drop(columns=[TARGET])

y_test = test_df[TARGET].copy()
X_test = test_df.drop(columns=[TARGET])

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")


X_train: (1672271, 189)   y_train: (1672271,)
X_test:  (418068, 189)   y_test:  (418068,)


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
10. Resumen y guardado de tablas
</h2>

</div>

In [9]:
resumen_gold = pd.DataFrame([
    {"conjunto": "Entrenamiento", "filas": X_train.shape[0], "columnas": X_train.shape[1]},
    {"conjunto": "Prueba", "filas": X_test.shape[0], "columnas": X_test.shape[1]},
])
display(resumen_gold)
guardar_tabla(resumen_gold, "ResumenGold")

guardar_tabla(y_train.describe().to_frame("Length of Stay (train)"), "EstadisticasObjetivoTrain")
guardar_tabla(y_test.describe().to_frame("Length of Stay (test)"), "EstadisticasObjetivoTest")


,conjunto,filas,columnas
0,Entrenamiento,1672271,189
1,Prueba,418068,189


Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_ResumenGold.csv
Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_EstadisticasObjetivoTrain.csv
Tabla guardada: C:\Users\danie\OneDrive - Universidad Externado de Colombia\Tesis\SPARCS\tablas\03_EstadisticasObjetivoTest.csv


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
11. Exportación
</h2>

</div>

In [10]:
X_train.to_parquet(GOLD_PATH / "X_train_sparcs.parquet", index=False)
X_test.to_parquet(GOLD_PATH / "X_test_sparcs.parquet", index=False)
y_train.to_frame("Length of Stay").to_parquet(GOLD_PATH / "y_train_sparcs.parquet", index=False)
y_test.to_frame("Length of Stay").to_parquet(GOLD_PATH / "y_test_sparcs.parquet", index=False)

print("=" * 60)
print("Conjuntos Gold (train/test) exportados correctamente")
print("=" * 60)
for nombre, obj in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test)]:
    print(f"{nombre}: {obj.shape}")


Conjuntos Gold (train/test) exportados correctamente
X_train: (1672271, 189)
X_test: (418068, 189)
y_train: (1672271,)
y_test: (418068,)


<div style="background:#BF360C;
padding:18px;
border-radius:8px;
color:white;
font-family:Arial,sans-serif">

<h2 style="margin:0;">
12. Conclusiones
</h2>

</div>

<div style="background:#FFF5F5;
border-left:6px solid #C62828;
padding:18px;
border-radius:0 0 8px 8px;
font-family:Arial,sans-serif">


El split 80/20 y el Target Encoding de las variables de alta cardinalidad ya quedaron resueltos en
esta etapa. <code>04_Modeling.ipynb</code> debe cargar directamente los 4 archivos exportados
(<code>X_train</code>, <code>X_test</code>, <code>y_train</code>, <code>y_test</code>) y
<b>no debe volver a hacer <code>train_test_split</code></b> — hacerlo de nuevo mezclaría datos que ya
fueron usados para calcular el Target Encoding, reintroduciendo el riesgo de fuga de información que
esta reestructuración buscaba evitar.


</div>